In [5]:
import pandas as pd
import numpy as np

df_1 = pd.read_excel('../data/raw/online_retail.xlsx', sheet_name='Year 2009-2010')
df_2 = pd.read_excel('../data/raw/online_retail.xlsx', sheet_name='Year 2010-2011')

# Combine into one dataframe
df = pd.concat([df_1, df_2], ignore_index=True)
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [7]:
print("DataFrame shape:", df.shape)
df.info()

DataFrame shape: (1067371, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


In [9]:
df.describe()
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

Based on the information of this data frame, it shows that out of 1067371 datapoints, there are around 243007 null data points for Customer ID variable. This could be due to technical issue while gathering the data, causing we unable to match the 243007 invoice to their specific customer's spending pattern. Moreover, there are also 4382 null values for Description.

In [10]:
print("Negative Quantity:", (df['Quantity'] < 0).sum())
print("Zero/Negative Price:", (df['Price'] <= 0).sum())

print("Duplicate rows:", df.duplicated().sum())


Negative Quantity: 22950
Zero/Negative Price: 6207
Duplicate rows: 34335


When we dig one step deeper in the data, there are 22950 negative quantities, 6207 zero or negative prices, and 34335 duplicated rows.

In [12]:
df[df['Description'].isnull()].head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom
6378,489882,35751C,NaN,12,2009-12-02 16:22:00,0.0,NaN,United Kingdom
6555,489898,79323G,NaN,954,2009-12-03 09:40:00,0.0,NaN,United Kingdom
6576,489901,21098,NaN,-200,2009-12-03 09:47:00,0.0,NaN,United Kingdom
6581,489903,21166,NaN,48,2009-12-03 09:57:00,0.0,NaN,United Kingdom


For the rows where description is null, we know that this could be system error since all of the key informations in errorenous, we will delete the rows. In general, the imputation decision for each invalid data are listed: 

Issue	Count	Decision
Null CustomerID	243,007	Drop — can't segment unknown customers
Null Description	4,382	Drop — likely system errors
Negative Quantity	22,950	Drop — these are returns/cancellations
Zero/Negative Price	6,207	Drop — free items or adjustments
Duplicates	34,335	Drop

**for the items with 0 price, I assume the store does not have sales event that give away free groceries, so they should not have free items in invoice.

In [13]:
cleaning_log = {"raw_rows": len(df)}

# 1. Drop null CustomerID
df = df.dropna(subset=['Customer ID'])
cleaning_log["after_drop_null_customer"] = len(df)

# 2. Drop null Description
df = df.dropna(subset=['Description'])
cleaning_log["after_drop_null_description"] = len(df)

# 3. Drop negative quantities (returns/cancellations)
df = df[df['Quantity'] > 0]
cleaning_log["after_drop_negative_quantity"] = len(df)

# 4. Drop zero/negative prices
df = df[df['Price'] >= 0]
cleaning_log["after_drop_zero_price"] = len(df)

# 5. Drop duplicates
df = df.drop_duplicates()
cleaning_log["after_drop_duplicates"] = len(df)

# Summary
for step, count in cleaning_log.items():
    print(f"{step}: {count}")

raw_rows: 1067371
after_drop_null_customer: 824364
after_drop_null_description: 824364
after_drop_negative_quantity: 805620
after_drop_zero_price: 805620
after_drop_duplicates: 779495


In [ ]:
df['TotalPrice'] = df['Quantity'] * df['Price']
df.head()


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [16]:
df.to_csv('../data/processed/online_retail_clean.csv', index=False)
print("Saved:", df.shape)

Saved: (779495, 9)
